# VGG输出漂移率：完整实现（解决SSL下载问题）

本notebook提供多种方式加载VGG16模型：
1. 使用预训练模型（需要下载）
2. 不使用预训练模型（随机初始化）
3. 从本地文件加载预训练模型

## 1. 导入必要的库

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw, ImageFont
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms, models
import torch.nn.functional as F
from sklearn.metrics import accuracy_score, classification_report
import os
import random
import ssl

# 设置随机种子
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

# 设备配置
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"使用设备: {device}")

## 2. 检查VGG16预训练模型是否已下载

In [ ]:
# 检查模型文件是否存在
cache_dir = os.path.expanduser('~/.cache/torch/hub/checkpoints')
model_path = os.path.join(cache_dir, 'vgg16-397923af.pth')

print("检查VGG16预训练模型...")
print(f"模型路径: {model_path}")

if os.path.exists(model_path):
    file_size = os.path.getsize(model_path) / 1024 / 1024
    print(f"✓ 已找到预训练模型 (大小: {file_size:.2f} MB)")
    USE_PRETRAINED = True
else:
    print("✗ 未找到预训练模型")
    print("\n选择加载方式:")
    print("  1. 尝试下载预训练模型（可能失败）")
    print("  2. 使用随机初始化（推荐，训练时间更长但不需要下载）")
    print()
    choice = input("请选择 (1/2，默认2): ").strip()
    
    if choice == '1':
        USE_PRETRAINED = True
        print("\n将尝试下载预训练模型...")
    else:
        USE_PRETRAINED = False
        print("\n将使用随机初始化模型")

## 3. Flanker刺激生成器

In [ ]:
class FlankerStimulusGenerator:
    """
    Flanker任务刺激生成器
    目标鸟和干扰鸟颜色相同（黑色）
    """
    
    def __init__(self, img_size=128, bird_size=20):
        self.img_size = img_size
        self.bird_size = bird_size
        self.directions = ['L', 'R', 'U', 'D']  # 左右上下
        self.direction_arrows = {
            'L': '◀',
            'R': '▶', 
            'U': '▲',
            'D': '▼'
        }
        
        # 布局定义：干扰项相对于目标的位置
        self.layouts = {
            'horizontal': [(-2, 0), (-1, 0), (1, 0), (2, 0)],
            'vertical': [(0, -2), (0, -1), (0, 1), (0, 2)],
            'cross': [(-1, 0), (1, 0), (0, -1), (0, 1)],
            'diagonal': [(-1, -1), (1, 1), (-1, 1), (1, -1)]
        }
        self.spacing = 25  # 鸟之间的间距
    
    def draw_bird(self, draw, center_x, center_y, direction):
        """
        绘制一个指向特定方向的鸟（用箭头表示）
        目标鸟和干扰鸟使用相同的颜色（黑色）
        """
        arrow = self.direction_arrows[direction]
        
        # 使用更大的字体绘制箭头
        try:
            font = ImageFont.truetype("/System/Library/Fonts/Helvetica.ttc", self.bird_size)
        except:
            try:
                font = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf", self.bird_size)
            except:
                font = ImageFont.load_default()
        
        # 获取文本边界框
        bbox = draw.textbbox((0, 0), arrow, font=font)
        text_width = bbox[2] - bbox[0]
        text_height = bbox[3] - bbox[1]
        
        # 计算绘制位置（居中）
        x = center_x - text_width // 2
        y = center_y - text_height // 2
        
        # 绘制箭头（黑色）
        draw.text((x, y), arrow, font=font, fill='black')
    
    def generate_stimulus(self, target_dir, flanker_dir, layout='horizontal', 
                      target_pos=None, bg_color='white'):
        """
        生成单个Flanker刺激图像
        """
        # 创建空白图像
        img = Image.new('RGB', (self.img_size, self.img_size), bg_color)
        draw = ImageDraw.Draw(img)
        
        # 设置目标位置（默认为中心）
        if target_pos is None:
            target_pos = (self.img_size // 2, self.img_size // 2)
        
        # 绘制目标（中心，黑色）
        self.draw_bird(draw, target_pos[0], target_pos[1], target_dir)
        
        # 计算干扰项位置
        offsets = self.layouts[layout]
        for offset in offsets:
            flanker_pos = (
                target_pos[0] + offset[0] * self.spacing,
                target_pos[1] + offset[1] * self.spacing
            )
            # 绘制干扰项（周围，也是黑色）
            self.draw_bird(draw, flanker_pos[0], flanker_pos[1], flanker_dir)
        
        return img
    
    def generate_dataset(self, n_samples=1000):
        """
        生成完整的Flanker数据集
        """
        images = []
        labels = []
        metadata = {
            'target_dirs': [],
            'flanker_dirs': [],
            'layouts': [],
            'positions': []
        }
        
        for i in range(n_samples):
            # 随机选择参数
            target_dir = random.choice(self.directions)
            flanker_dir = random.choice(self.directions)
            layout = random.choice(list(self.layouts.keys()))
            
            # 判断一致性
            is_congruent = (target_dir == flanker_dir)
            label = 0 if is_congruent else 1
            
            # 生成图像
            img = self.generate_stimulus(target_dir, flanker_dir, layout)
            
            # 转换为numpy数组并归一化
            img_array = np.array(img).astype(np.float32) / 255.0
            # 转换为CHW格式 (Channels, Height, Width)
            img_array = np.transpose(img_array, (2, 0, 1))
            
            images.append(img_array)
            labels.append(label)
            
            # 保存元数据
            metadata['target_dirs'].append(target_dir)
            metadata['flanker_dirs'].append(flanker_dir)
            metadata['layouts'].append(layout)
            metadata['positions'].append((self.img_size//2, self.img_size//2))
        
        return np.array(images), np.array(labels), metadata

print("Flanker刺激生成器定义完成")

## 4. 四个方向漂移率的详细物理意义

In [ ]:
print("="*80)
print("四个方向漂移率的详细物理意义")
print("="*80)
print()

print("【输出格式】")
print("─"*80)
print("模型输出: [漂移率_左, 漂移率_右, 漂移率_上, 漂移率_下]")
print("索引:     [    0,      1,      2,      3]")
print("方向:     [   L,      R,      U,      D]")
print()

print("【核心概念】漂移率 = 信息累积速度")
print("─"*80)
print()

print("在LBA（Linear Ballistic Accumulator）模型中：")
print("  - 每个决策选项对应一个累加器")
print("  - 累加器从起始点开始累积证据")
print("  - 漂移率决定累积速度")
print("  - 最快到达阈值的累加器获胜")
print()

print("【四个方向的具体含义】")
print("─"*80)
print()

print("漂移率[0] - 左方向 (Left, L)")
print("  物理意义：支持'向左'决策的证据累积速度")
print("  高值 (>2.0): 视觉刺激强烈支持向左")
print("  低值 (<1.0): 视觉刺激不支持向左")
print("  单位: 信息单位/时间单位 (如bits/second)")
print()

print("漂移率[1] - 右方向 (Right, R)")
print("  物理意义：支持'向右'决策的证据累积速度")
print("  与左方向竞争：如果左和右都高，则存在冲突")
print()

print("漂移率[2] - 上方向 (Up, U)")
print("  物理意义：支持'向上'决策的证据累积速度")
print("  垂直方向的竞争")
print()

print("漂移率[3] - 下方向 (Down, D)")
print("  物理意义：支持'向下'决策的证据累积速度")
print("  与上方向竞争")
print()

print("【数值大小的具体含义】")
print("─"*80)
print()
print("示例：漂移率 = [2.5, 0.3, 0.2, 0.1]")
print()
print("  2.5 (左方向)：")
print("    - 非常高的累积速度")
print("    - 视觉刺激强烈支持向左")
print("    - 预期：快速选择左方向")
print("    - 反应时间：短 (RT ≈ 0.4秒)")
print("    - 置信度：高")
print()
print("  0.3 (右方向)：")
print("    - 较低的累积速度")
print("    - 视觉刺激不支持向右")
print("    - 预期：不太可能选择右方向")
print()
print("  0.2 (上方向)：")
print("    - 很低的累积速度")
print("    - 几乎不支持向上")
print()
print("  0.1 (下方向)：")
print("    - 非常低的累积速度")
print("    - 几乎不支持向下")

## 5. VGG模型输出漂移率的完整实现

In [ ]:
class VGGDriftRateModel(nn.Module):
    """
    VGG模型输出4个漂移率
    
    架构：
    1. VGG16特征提取器
    2. 自定义回归头输出4个漂移率
    3. 使用ReLU确保漂移率为正值
    """
    
    def __init__(self, use_pretrained=True):
        super(VGGDriftRateModel, self).__init__()
        
        print(f"\n初始化VGG16模型 (预训练={use_pretrained})...")
        
        try:
            if use_pretrained:
                # 尝试加载预训练模型
                try:
                    # 使用新的API
                    vgg = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1)
                    print("✓ 成功加载预训练VGG16模型")
                except Exception as e:
                    print(f"✗ 加载预训练模型失败: {e}")
                    print("  使用随机初始化...")
                    vgg = models.vgg16(weights=None)
            else:
                # 使用随机初始化
                vgg = models.vgg16(weights=None)
                print("✓ 使用随机初始化VGG16模型")
        except Exception as e:
            print(f"✗ 创建VGG16失败: {e}")
            raise
        
        # 特征提取部分
        self.features = vgg.features
        self.avgpool = vgg.avgpool
        
        # 计算特征维度
        self.feature_dim = 512 * 4 * 4  # VGG16在128x128输入下的特征维度
        
        # 漂移率输出头
        # 输出4个漂移率：[左, 右, 上, 下]
        self.drift_head = nn.Sequential(
            nn.Linear(self.feature_dim, 1024),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(1024, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(512, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(256, 4),  # 输出4个漂移率
            nn.ReLU()  # 确保漂移率为正值
        )
        
        # 初始化最后一层的偏置，使初始漂移率在合理范围
        nn.init.constant_(self.drift_head[-2].bias, 0.5)
        
        print(f"✓ 模型初始化完成")
        print(f"  特征维度: {self.feature_dim}")
        print(f"  输出维度: 4 (左, 右, 上, 下)")
        
    def forward(self, x):
        """
        前向传播
        
        参数:
            x: 输入图像 (batch_size, 3, 128, 128)
            
        返回:
            drift_rates: 漂移率 (batch_size, 4)
                        [左, 右, 上, 下]
        """
        # 特征提取
        x = self.features(x)  # (batch, 512, 4, 4)
        x = self.avgpool(x)   # (batch, 512, 4, 4)
        
        # 展平
        x = torch.flatten(x, 1)  # (batch, 8192)
        
        # 输出漂移率
        drift_rates = self.drift_head(x)  # (batch, 4)
        
        return drift_rates
    
    def predict_choice(self, drift_rates):
        """
        根据漂移率预测选择
        
        参数:
            drift_rates: 漂移率 (batch_size, 4)
            
        返回:
            choices: 预测的选择 (batch_size,)
                    0=左, 1=右, 2=上, 3=下
        """
        return torch.argmax(drift_rates, dim=1)
    
    def predict_rt(self, drift_rates, threshold=1.0):
        """
        根据漂移率预测反应时间（简化版）
        
        参数:
            drift_rates: 漂移率 (batch_size, 4)
            threshold: 决策阈值
            
        返回:
            rts: 预测的反应时间 (batch_size,)
        """
        # 找到最高漂移率
        max_drifts, _ = torch.max(drift_rates, dim=1)
        
        # 简化的RT预测：RT = threshold / drift_rate
        rts = threshold / (max_drifts + 1e-6)
        
        return rts

# 创建模型
print("\n创建VGG漂移率模型...")
model = VGGDriftRateModel(use_pretrained=USE_PRETRAINED).to(device)

# 统计参数
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\n模型统计:")
print(f"  总参数: {total_params:,}")
print(f"  可训练参数: {trainable_params:,}")

## 6. 测试模型输出

In [ ]:
# 生成测试数据
print("生成测试数据...")
generator = FlankerStimulusGenerator(img_size=128, bird_size=24)
test_images, test_labels, test_metadata = generator.generate_dataset(n_samples=100)

# 转换为PyTorch张量
test_tensor = torch.from_numpy(test_images).float().to(device)

# 前向传播
model.eval()
with torch.no_grad():
    drift_rates = model(test_tensor[:10])
    
# 打印输出
print("\n" + "="*80)
print("模型输出示例")
print("="*80)
print()

direction_names = ['左', '右', '上', '下']

for i in range(5):
    print(f"样本 {i+1}:")
    print(f"  目标方向: {test_metadata['target_dirs'][i]}")
    print(f"  干扰方向: {test_metadata['flanker_dirs'][i]}")
    print(f"  一致性: {'Congruent' if test_labels[i]==0 else 'Incongruent'}")
    print(f"  漂移率: {drift_rates[i].cpu().numpy()}")
    print(f"  解读:")
    for j, name in enumerate(direction_names):
        print(f"    {name}方向: {drift_rates[i][j].item():.3f}")
    print()

## 7. 漂移率与认知过程的关联

In [ ]:
print("="*80)
print("漂移率与认知过程的详细关联")
print("="*80)
print()

print("【1. 漂移率与反应时间的关系】")
print("─"*80)
print()
print("公式：RT ≈ threshold / drift_rate")
print()
print("示例：")
print("  假设阈值 = 1.0")
print("  漂移率 = [2.5, 0.3, 0.2, 0.1]")
print()
print("  预测RT：")
print("    左方向: RT = 1.0 / 2.5 = 0.4 秒")
print("    右方向: RT = 1.0 / 0.3 = 3.3 秒")
print("    上方向: RT = 1.0 / 0.2 = 5.0 秒")
print("    下方向: RT = 1.0 / 0.1 = 10.0 秒")
print()
print("  结果：左方向最快到达阈值 → 选择左方向")
print("  预测RT：约0.4秒")
print()

print("【2. 漂移率与选择概率的关系】")
print("─"*80)
print()
print("使用Softmax计算选择概率：")
print("  P(选择i) = exp(drift_i) / Σ exp(drift_j)")
print()
print("示例：")
print("  漂移率 = [2.5, 0.3, 0.2, 0.1]")
print("  Softmax概率：")
print("    P(左) = exp(2.5) / (exp(2.5)+exp(0.3)+exp(0.2)+exp(0.1))")
print("          = 12.18 / (12.18+1.35+1.22+1.11)")
print("          = 12.18 / 15.86")
print("          = 0.768 (76.8%)")
print("    P(右) = 1.35 / 15.86 = 0.085 (8.5%)")
print("    P(上) = 1.22 / 15.86 = 0.077 (7.7%)")
print("    P(下) = 1.11 / 15.86 = 0.070 (7.0%)")
print()
print("  结果：选择左方向的概率最高")
print()

print("【3. 漂移率与一致性判断的关系】")
print("─"*80)
print()
print("Congruent刺激：")
print("  特征：漂移率高度集中")
print("  示例：[2.8, 0.2, 0.1, 0.1]")
print("  集中度：2.8 / 0.2 = 14.0")
print("  判断：高集中度 → Congruent")
print()
print("Incongruent刺激：")
print("  特征：漂移率相对分散")
print("  示例：[2.1, 1.7, 0.3, 0.2]")
print("  集中度：2.1 / 1.7 = 1.24")
print("  判断：低集中度 → Incongruent")

## 8. 可视化漂移率分布

In [ ]:
# 选择Congruent和Incongruent样本
congruent_idx = np.where(test_labels == 0)[0][:5]
incongruent_idx = np.where(test_labels == 1)[0][:5]

# 获取漂移率
model.eval()
with torch.no_grad():
    congruent_drifts = model(test_tensor[congruent_idx]).cpu().numpy()
    incongruent_drifts = model(test_tensor[incongruent_idx]).cpu().numpy()

# 可视化
fig, axes = plt.subplots(2, 5, figsize=(20, 8))

direction_names = ['左', '右', '上', '下']
x_pos = np.arange(4)

# Congruent样本
for i in range(5):
    axes[0, i].bar(x_pos, congruent_drifts[i], color='green', alpha=0.7)
    axes[0, i].set_xlabel('方向')
    axes[0, i].set_ylabel('漂移率')
    axes[0, i].set_title(f'Congruent样本{i+1}\n目标:{test_metadata["target_dirs"][congruent_idx[i]]}')
    axes[0, i].set_xticks(x_pos)
    axes[0, i].set_xticklabels(direction_names)
    axes[0, i].set_ylim([0, 3.0])

# Incongruent样本
for i in range(5):
    axes[1, i].bar(x_pos, incongruent_drifts[i], color='orange', alpha=0.7)
    axes[1, i].set_xlabel('方向')
    axes[1, i].set_ylabel('漂移率')
    axes[1, i].set_title(f'Incongruent样本{i+1}\n目标:{test_metadata["target_dirs"][incongruent_idx[i]]}, 干扰:{test_metadata["flanker_dirs"][incongruent_idx[i]]}')
    axes[1, i].set_xticks(x_pos)
    axes[1, i].set_xticklabels(direction_names)
    axes[1, i].set_ylim([0, 3.0])

plt.tight_layout()
plt.show()

print("注意：这些是未训练模型的输出，漂移率分布是随机的")
print("训练后，模型会学习到正确的漂移率模式")

## 9. 总结

In [ ]:
print("="*80)
print("总结：四个方向漂移率的完整意义")
print("="*80)
print()

print("【输出格式】")
print("─"*80)
print("模型输出: [漂移率_左, 漂移率_右, 漂移率_上, 漂移率_下]")
print("索引:     [    0,      1,      2,      3]")
print("方向:     [   L,      R,      U,      D]")
print()

print("【每个数值的具体含义】")
print("─"*80)
print()

print("漂移率[i] = 该方向的信息累积速度")
print()
print("物理意义：")
print("  1. 决策证据的累积速度")
print("  2. 单位时间内累积的信息量")
print("  3. 对该方向决策的支持强度")
print()

print("数值解释：")
print("  - 高值 (>2.0): 强烈支持该方向")
print("  - 中值 (1.0-2.0): 中等支持")
print("  - 低值 (<1.0): 弱支持或不支持")
print()

print("【四个方向的相互关系】")
print("─"*80)
print()

print("竞争关系：")
print("  - 左 vs 右：水平方向的竞争")
print("  - 上 vs 下：垂直方向的竞争")
print("  - 四个方向相互竞争，最高者获胜")
print()

print("决策机制：")
print("  - 每个方向对应一个累加器")
print("  - 累加器以漂移率的速度累积证据")
print("  - 最快到达阈值的累加器获胜")
print("  - 决定最终的选择和反应时间")
print()

print("【应用示例】")
print("─"*80)
print()

print("示例1：Congruent刺激（所有箭头向左）")
print("  漂移率: [2.8, 0.2, 0.1, 0.1]")
print("  解读:")
print("    - 左方向(2.8): 非常高的支持")
print("    - 其他方向(<0.3): 几乎不支持")
print("    - 预测: 快速选择左方向")
print("    - RT: 短 (约0.36秒)")
print()

print("示例2：Incongruent刺激（目标向左，干扰向右）")
print("  漂移率: [2.1, 1.7, 0.3, 0.2]")
print("  解读:")
print("    - 左方向(2.1): 高支持（目标）")
print("    - 右方向(1.7): 较高支持（干扰）")
print("    - 存在竞争: 左vs右")
print("    - 预测: 选择左方向（但较慢）")
print("    - RT: 较长 (约0.48秒)")